In [ ]:
from libraries import *
from parameters import *
import pegasusio as io

In [ ]:
%load_ext rpy2.ipython

In [ ]:
os.getcwd()
os.chdir(projectDir)

In [ ]:
conf_mt_prefix = 'MT-' if par_species == 'human' else 'mt-'

In [ ]:
conf_samples = {}

for elem in ["SAM24453279", "SAM24453280"]:
    print(elem)
    ad = sc.read_10x_h5("./Data/"+elem+"_filtered_feature_bc_matrix.h5")
    ad.var_names_make_unique()
    print(ad.shape)
    outs  = io.read_input("./Data/"+elem+"_outputs_demux.zarr.zip")
    obs = outs.obs
    
    obs.index = [str(x)+"-1" for x in obs.index]
    obs = obs.loc[[x for x in obs.index if x in ad.obs.index]]
    ad = ad[[x for x in ad.obs.index if x in obs.index],:]
    obs = obs.loc[ad.obs.index]
    ad.obs = ad.obs.join(obs,how="inner")
    
    
    ad.obs['n_umis']  = ad.X.sum(1)
    ad.obs['n_genes'] = (ad.X != 0).sum(1).A1
    ad.obs['log10_n_umis'] = np.log10(ad.X.sum(1))
    ad.obs['log10_n_genes'] = np.log10((ad.X != 0).sum(1).A1)
    ad.obs['sample_name'] = elem
    #ad.obs.index = [str(x)+"_"+str(elem) for x in ad.obs.index]
    
    mt_gene_mask = ad.var_names.str.startswith(conf_mt_prefix)
    assert mt_gene_mask.sum() > 0, 'Wrong mt prefix'
    ad.obs['mt_frac'] = ad.X[:, mt_gene_mask].sum(1).A1 / ad.obs['n_umis']

    conf_samples[elem] = ad.copy()
  

In [ ]:
conf_samples

In [ ]:
batch_categories, ads = zip(*conf_samples.items())

adata = sc.AnnData.concatenate(*ads, join="inner", 
                               batch_key=par_batch_key,
                               batch_categories=batch_categories)


In [ ]:
adata.obs["demux_type"].value_counts()

In [ ]:
adata = adata[adata.obs["demux_type"] == "singlet",:]
adata = adata[adata.obs["mt_frac"] < 0.2,:]
adata = adata[adata.obs["n_genes"] < 5000,:]
adata = adata[adata.obs["n_genes"] > 800,:]

In [ ]:
sc.pp.filter_genes(adata, min_cells=100)

In [ ]:
adata

In [ ]:
adata.obs["assignment"] = adata.obs["assignment"].astype(str)
adata.obs["sample_name"] = adata.obs["sample_name"].astype(str)
adata.obs["HTO_sample"] = adata.obs["assignment"] + "_" + adata.obs["sample_name"]

In [ ]:
adata.obs["Sample_type"] = "NTC"

adata.obs.loc[adata.obs["HTO_sample"] == "HTO09_SAM24453279","Sample_type"]="NTC_CSFBS"
adata.obs.loc[adata.obs["HTO_sample"] == "HTO10_SAM24453279","Sample_type"]="NTC_CoCAFS"
adata.obs.loc[adata.obs["HTO_sample"] == "HTO11_SAM24453279","Sample_type"]="PRbPt"
adata.obs.loc[adata.obs["HTO_sample"] == "HTO12_SAM24453280","Sample_type"]="PRbPt_CSFBS"
adata.obs.loc[adata.obs["HTO_sample"] == "HTO13_SAM24453280","Sample_type"]="PRbPt_CoCAFS"
adata.obs.loc[adata.obs["HTO_sample"] == "HTO14_SAM24453280","Sample_type"]="CAFS_NTC"
adata.obs.loc[adata.obs["HTO_sample"] == "HTO15_SAM24453280","Sample_type"]="CAFS_PRbPt"


In [ ]:
adata.obs["Sample_type"].value_counts()

In [ ]:
adata = adata[adata.obs["Sample_type"] != "CAFS_NTC",]
adata = adata[adata.obs["Sample_type"] != "CAFS_PRbPt",]

In [ ]:
adata.layers['counts'] = adata.X.copy()

sc.pp.normalize_total(adata, target_sum=par_preprocessing_target_sum)
sc.pp.log1p(adata)
adata.raw = adata

In [ ]:
gene_list_url = 'https://raw.githubusercontent.com/theislab/scanpy_usage/master/180209_cell_cycle/data/regev_lab_cell_cycle_genes.txt'

cell_cycle_genes = [str(x.strip(), 'utf-8').capitalize() for x in urlopen(gene_list_url)] # capitalize = shame


s_genes = cell_cycle_genes[:43]
g2m_genes = cell_cycle_genes[43:]


sc.tl.score_genes_cell_cycle(adata, s_genes=s_genes, g2m_genes=g2m_genes)

In [ ]:
sc.pp.regress_out(adata, ['S_score','G2M_score','log10_n_umis', 'mt_frac', 'n_genes'], n_jobs=30)

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=par_downstream_n_top_genes)
sc.pp.scale(adata, max_value=10)
sc.pp.pca(adata, n_comps=50, svd_solver='arpack')
sc.pp.neighbors(adata, n_neighbors=10)
sc.tl.umap(adata)

In [ ]:
adata.write("./outputs/anndata/AbbasAnndata.h5ad")

In [ ]:
from matplotlib.pyplot import rc_context

with rc_context({'figure.figsize': (8, 3)}):
    sc.pl.violin(adata, ['mt_frac'], 
                stripplot=False, inner='box', groupby='Sample_type')

In [ ]:
from matplotlib.pyplot import rc_context

with rc_context({'figure.figsize': (12, 3)}):
    sc.pl.violin(adata, ['n_genes'], 
                stripplot=False, inner='box', groupby='Sample_type')  # use stripplot=False to remove the internal dots, inner='box' adds a boxplot inside violins


In [ ]:
sc.tl.leiden(adata, resolution=0.5)

f, ax = plt.subplots(figsize=(4, 4))
sc.pl.umap(adata, color='leiden', legend_loc='on data', 
           legend_fontoutline=3, legend_fontsize=14, 
           legend_fontweight='normal', title='Clusters', ax=ax, show=False, size=8);

In [ ]:
f, ax = plt.subplots(figsize=(4, 4))
sc.pl.umap(adata, color='mt_frac', legend_loc='on data', 
           legend_fontoutline=3, legend_fontsize=14, 
           legend_fontweight='normal', title='Clusters', ax=ax, show=False, size=8);

In [ ]:
f, ax = plt.subplots(figsize=(4, 4))
sc.pl.umap(adata, color='n_genes', legend_loc='on data', 
           legend_fontoutline=3, legend_fontsize=14, 
           legend_fontweight='normal', title='Clusters', ax=ax, show=False, size=8);

In [ ]:
adata.obs["Sample_type"] = adata.obs["Sample_type"].astype("category")

In [ ]:
f, ax = plt.subplots(figsize=(6, 6))

sc.pl.umap(adata, color='Sample_type', 
           legend_fontoutline=3, legend_fontsize=14, ax=ax, 
           legend_fontweight='normal', title='Clusters', size=18);

In [ ]:
f, ax = plt.subplots(figsize=(6, 6))

sc.pl.umap(adata, color='leiden', 
           legend_fontoutline=3, legend_fontsize=14, ax=ax, 
           legend_fontweight='normal', title='Clusters');

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="leiden", n_genes=2000, method="t-test_overestim_var")
sc.tl.dendrogram(adata, groupby='leiden')
sc.pl.rank_genes_groups_matrixplot(adata, n_genes=10, standard_scale='var', cmap='Blues')

In [ ]:
markerGenes = pd.DataFrame(adata.uns['rank_genes_groups']['names'])
markerGenes = markerGenes.iloc[0:40,:]
markerGenes.to_csv("./TextFiles/Leiden_markers_withoutCAFS.csv")

In [ ]:
for i in  markerGenes.columns:
    myGeneList = [x for x in markerGenes.loc[:,i] if x != ' ']
    myGeneList = [x for x in markerGenes.loc[:,i] if x !="nan"]
    myGeneList = [x for x in markerGenes.loc[:,i] if not isinstance(x, float)]

    myGeneList = [x.replace(".","-") for x in myGeneList]

    print(myGeneList)
    if(sum(pd.Series(myGeneList).isin(adata.var_names)) > 1):
        sc.tl.score_genes(adata=adata, gene_list=myGeneList, score_name=i)
        sc.pl.umap(adata, color=i, size=10, color_map="coolwarm")
        f, ax = plt.subplots(figsize=(12, 4))
        sc.pl.violin(adata, i, groupby='Sample_type', ax=ax)

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="Sample_type", n_genes=2000, method="t-test_overestim_var")
sc.tl.dendrogram(adata, groupby='Sample_type')
sc.pl.rank_genes_groups_matrixplot(adata, n_genes=10, standard_scale='var', cmap='Blues')

In [ ]:
sc.tl.rank_genes_groups(adata, 'Sample_type', groups=['PRbPt_CoCAFS'], reference='NTC_CoCAFS', method='wilcoxon')

In [ ]:
dedf = sc.get.rank_genes_groups_df(adata, group="PRbPt_CoCAFS")

In [ ]:
dedf.to_csv("./TextFiles/DEGenes_PRbPt_CoCAFS_NTC_CoCAFS.csv")

In [ ]:
adata.uns['rank_genes_groups']

In [ ]:
markerGenesSamples = pd.DataFrame(adata.uns['rank_genes_groups']['names'])
markerGenesSamples = markerGenesSamples.iloc[0:40,:]


In [ ]:
markerGenesSamples.to_csv("./TextFiles/MarkerGenesSamples_withoutCAFS.csv")

In [ ]:
for i in  markerGenesSamples.columns:
    
    myGeneList = [x for x in markerGenesSamples.loc[:,i] if x != ' ']
    myGeneList = [x for x in markerGenesSamples.loc[:,i] if x !="nan"]
    myGeneList = [x for x in markerGenesSamples.loc[:,i] if not isinstance(x, float)]

    myGeneList = [x.replace(".","-") for x in myGeneList]

    print(myGeneList)
    if(sum(pd.Series(myGeneList).isin(adata.var_names)) > 1):
        sc.tl.score_genes(adata=adata, gene_list=myGeneList, score_name=i)
        sc.pl.umap(adata, color=i, size=10, color_map="coolwarm")
        f, ax = plt.subplots(figsize=(12, 4))
        sc.pl.violin(adata, i, groupby='Sample_type', ax=ax)

In [ ]:
geneSignatures = pd.DataFrame(pd.read_csv("./TextFiles/Abbas_geneSignatures.csv"))

In [ ]:
geneSignatures

In [ ]:
for i in  geneSignatures.columns:
    print(i)
    myGeneList = [x for x in geneSignatures.loc[:,i] if x != ' ']
    myGeneList = [x for x in geneSignatures.loc[:,i] if x !="nan"]
    myGeneList = [x for x in geneSignatures.loc[:,i] if not isinstance(x, float)]

    myGeneList = [x.replace(".","-") for x in myGeneList]

    print(myGeneList)
    print(sum(pd.Series(myGeneList).isin(adata.var_names)))
    if(sum(pd.Series(myGeneList).isin(adata.var_names)) > 1):
        sc.tl.score_genes(adata=adata, gene_list=myGeneList, score_name=i)
        sc.pl.umap(adata, color=i, size=10, color_map="coolwarm", vmax=0.3, vmin=-0.3)
        f, ax = plt.subplots(figsize=(12, 4))
        sc.pl.violin(adata, i, groupby='Sample_type', ax=ax)

In [ ]:
enDistMat = pd.DataFrame(np.zeros(shape=(len(adata.obs["Sample_type"].unique()), len(adata.obs["Sample_type"].unique()))))
cond = list(adata.obs["Sample_type"].unique())
cond.sort()
enDistMat.index = cond
enDistMat.columns = cond


In [ ]:
from geomloss import SamplesLoss
import torch
Loss =  SamplesLoss("energy")

for i in cond:
    ad_rna_i = adata[adata.obs["Sample_type"]==i,:]
    for j in cond:
        ad_rna_j = adata[adata.obs["Sample_type"]==j,:]
        
        enDistMat.loc[i,j] = Loss( torch.from_numpy(ad_rna_i.obsm["X_pca"]), torch.from_numpy(ad_rna_j.obsm["X_pca"]) ).item()       

In [ ]:
%%R -i enDistMat -w 6 -h 6 -u in

library(pheatmap)

pheatmap(enDistMat, method="ward.D", cluster_rows= TRUE, cluster_cols=TRUE)